In [ ]:
import mysql.connector
import warnings
warnings.filterwarnings('ignore')
db_name="orderdb"
mydb_connection =mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database=db_name
    )

my_cursor=mydb_connection.cursor()


query="""
SELECT * from orders;
"""
# my_cursor.execute(query)
import pandas as pd
df=pd.read_sql_query(query,mydb_connection)
df.head()

,Order_id,Customer_code,Placed_at,Restaurant_id,Cuisine,Order_status,Promo_code_Name
0,OF1900191801,UFDDN1991918XUY1,2025-01-01 15:30:20,KMKMH6787,Lebanese,Delivered,Tasty50
1,OF1900191802,UFDDN1991918XUY1,2025-01-02 12:15:45,LEBANESE2,Lebanese,Delivered,NaN
2,OF1900191803,UFDDN1991918XUY1,2025-01-10 18:45:30,PIZZA123,Italian,Cancelled,HUNGRY20
3,OF1900191804,UFDDN1991918XUY1,2025-01-15 19:20:15,ITALIAN2,Italian,Delivered,NaN
4,OF1900191805,UFDDN1991918XUY1,2025-01-20 11:30:00,BURGER99,American,Delivered,NaN


**1.Top 1 outlets by cusine type without using limit and top**

In [10]:
query="""
with cte as (
    select Cuisine,Restaurant_id,
    count(*) as order_count
    from orders
    group by Cuisine,Restaurant_id)
select * from (
    select *,
    row_number() over(partition by Cuisine order by order_count desc) as rn
    from cte) a
where a.rn=1;
"""
pd.read_sql_query(query,mydb_connection)

,Cuisine,Restaurant_id,order_count,rn
0,American,BURGER99,8,1
1,Italian,PIZZA123,10,1
2,Japanese,SUSHI456,6,1
3,Lebanese,KMKMH6787,10,1
4,Mexican,TACO789,7,1


**2.find the daily new customer count from the launch date(everyday how many new customers are we aquiring)**

In [12]:
query="""
with cte as(
	select Customer_code,
	min(date(Placed_at)) as first_date
	from orders
	group by Customer_code)
select first_date ,
count(*) as customer_count
from cte
group by first_date;
"""
df=pd.read_sql_query(query,mydb_connection)
df.head(10)

,first_date,customer_count
0,2025-01-01,2
1,2025-01-02,1
2,2025-01-03,1
3,2025-01-04,1
4,2025-01-05,3
5,2025-01-06,1
6,2025-01-07,1
7,2025-01-08,1
8,2025-01-09,1
9,2025-01-10,3


**3.Count All the users who were aquired in jan and only place
<br>one order in jan and did not place any other order**

In [14]:
query="""
SELECT Customer_code, count(*) as orders
from orders
where MONTH(Placed_at)=1 and YEAR(Placed_at)=2025
and Customer_code not in(
	select distinct Customer_code
	from orders 
	where not (MONTH(Placed_at)=1 and YEAR(Placed_at)=2025)
    )
group by Customer_code 
having count(*)=1;
"""
df=pd.read_sql_query(query,mydb_connection)
df.head(10)

,Customer_code,orders
0,GHI5678901234XYZ,1
1,JKL3456789012XYZ,1
2,MNO7890123456XYZ,1
3,PQR1234567890ABC,1
4,STU9876543210ABC,1
5,VWX5678901234ABC,1
6,YZA3456789012ABC,1
7,BCD7890123456ABC,1
8,EFG1234567890DEF,1
9,HIJ9876543210DEF,1


**List all customers who made their first order using a promo code and have not placed any order in the last 7 days (based on the latest date available in the data).**

In [18]:
query="""
with customer_orders as(
	select Customer_code,
	min(Placed_at) as first_order_date,
	max(Placed_at) as last_order_date
	from orders
	group by Customer_code),
max_date as(
	select max(Placed_at) as ref_date
	from orders
)
select Customer_code,Date(first_order_date) as first_order_date,
Date(last_order_date) as last_order_date,
Promo_code_Name from (
	select c.Customer_code,first_order_date,
	last_order_date,o.Promo_code_Name
	from orders o
	join customer_orders c
	on o.Customer_code=c.Customer_code
	and o.Placed_at=c.first_order_date
	where o.Promo_code_name is Not NULL)a
    cross join max_date
where datediff(max_date.ref_date,a.last_order_date)>7;
"""
df=pd.read_sql_query(query,mydb_connection)
df.head(10)

,Customer_code,first_order_date,last_order_date,Promo_code_Name
0,ABC1234567890XYZ,2025-01-01,2025-01-05,NEWUSER
1,DEF9876543210XYZ,2025-01-02,2025-03-02,FIRSTORDER
2,GHI5678901234XYZ,2025-01-03,2025-01-03,NEWUSER
3,JKL3456789012XYZ,2025-01-04,2025-01-04,FIRSTORDER
4,PQR1234567890ABC,2025-01-06,2025-01-06,NEWUSER
5,VWX5678901234ABC,2025-01-08,2025-01-08,FIRSTORDER
6,BCD7890123456ABC,2025-01-10,2025-01-10,NEWUSER
7,HIJ9876543210DEF,2025-01-12,2025-01-12,FIRSTORDER
8,QRS7890123456DEF,2025-01-15,2025-01-15,NEWUSER
9,WXY9876543210GHI,2025-01-17,2025-01-17,FIRSTORDER
